In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pyarrow.parquet as pq
import pandas as pd 
from pathlib import Path
from PIL import Image
import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

In [15]:
PARQUET_PATH = Path("/Users/mainframe/Workspace/Graduate/notebooks/plant_gotchi_dataset.parquet")
PHOTO_PATH = Path("/Users/mainframe/Workspace/Graduate/dataset/photos")

df = pq.read_table(source=PARQUET_PATH, use_threads=True).to_pandas()

In [16]:
class PlantaPipeline:
    def __init__(self, size=512):
        # This is our 'Transformation Contract'
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            # Standard ImageNet normalization (works well for most lettuce)
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])

    def process(self, image_path: Path):
        """Reads and transforms a single image for the M4 Max NPU."""
        try:
            # 1. Open the image
            img = Image.open(image_path).convert('RGB')
            
            # 2. Apply the transformations
            img_tensor = self.transform(img)
            
            # 3. Add a batch dimension (C, H, W) -> (1, C, H, W)
            # Neural networks expect a 'batch' even if it's just one image
            return img_tensor.unsqueeze(0)
            
        except Exception as e:
            print(f"Failed to process {image_path.name}: {e}")
            return None

In [17]:
# Usage
pipeline = PlantaPipeline(size=512)
# Using your Path objects from the earlier step
sample_img = pipeline.process(PHOTO_PATH / "C30_L01_01_001_454963.jpg")

if sample_img is not None:
    print(f"Tensor Shape: {sample_img.shape}") # Expect: torch.Size([1, 3, 512, 512])

Tensor Shape: torch.Size([1, 3, 512, 512])


In [18]:
class PlantaDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe
        self.img_dir = Path(img_dir)
        self.transform = transform
        
        # Prepare tabular features for the MLP branch
        # We normalize these so the model doesn't choke on large numbers
        self.tabular_cols = ['temp', 'humid', 'co2', 'light', 'stem_area', 'leaf_area']
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 1. Get the row data
        row = self.df.iloc[idx]
        
        # 2. Handle the Image
        # Assuming your file_name needs an extension like .jpg
        img_path = self.img_dir / f"{row['file_name']}.jpg"
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        # 3. Handle the Tabular Data (MLP Input)
        # Convert the row features into a FloatTensor
        tab_data = torch.tensor(row[self.tabular_cols].values.astype('float32'))
        
        # 4. Handle the Label (Target)
        # Convert 'growth' (1, 2, 3) to a float for Regression
        label = torch.tensor([float(row['growth'])])
        
        return image, tab_data, label

In [19]:
train_ds = PlantaDataset(
    dataframe=df, 
    img_dir=PHOTO_PATH, 
    transform=pipeline.transform # Using the pipeline we built earlier
)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)

In [20]:
#test
images, tabular, labels = next(iter(train_loader))

In [12]:
class SimplePlantaCNN(nn.Module):
    def __init__(self):
        super(SimplePlantaCNN, self).__init__()
        
        # Block 1: 512 -> 256
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        
        # Block 2: 256 -> 128
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        
        # Block 3: 128 -> 64
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        
        # Global Average Pooling: Takes (64, 64, 64) -> (64, 1, 1)
        # This makes the model "resolution-independent"
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        
        # Regression Head
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 1) # Single output for Age/Stage regression

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        
        x = self.gap(x)
        x = torch.flatten(x, 1) # Flatten to (Batch, 64)
        
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialization on M4 Max
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = SimplePlantaCNN().to(device)
print(f"Model initialized on: {device}")

Model initialized on: mps
